# The Ninefold Tide — premium original character voice auditions

**This notebook is not a completed voice cast.** It creates original synthetic voice candidates and displays the WAVs for informed review. No paid TTS API. Requires a Kaggle GPU and internet enabled (Settings → Accelerator and Internet). The first Qwen checkpoint downloads are large. Stop GPU when finished. Do not publish audio without reading the licence and listening to the entire take.

In [ ]:
import os, sys, subprocess, pathlib
from IPython.display import display, Audio
assert subprocess.run(["bash", "-lc", "command -v nvidia-smi"], capture_output=True).returncode == 0, "Turn on a GPU in Kaggle settings first"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], capture_output=True, text=True).stdout)
# Cache weights ON KAGGLE storage, not on the user's phone
os.environ["HF_HOME"] = "/kaggle/temp/ninefold_hf_cache"\nprint("Model weights cache: /kaggle/temp (not included in saved /kaggle/working outputs)")
ROOT = pathlib.Path("/kaggle/working/TOOLS")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/eghosa001/TOOLS.git", str(ROOT)], check=True)
os.chdir(ROOT)
print("Repository:", ROOT)


## 1. Install the free inference engine on Kaggle

This step downloads Python dependencies to the Kaggle runtime. Qwen model weights are downloaded only in a later step with explicit permission.

**Storage protection:** Kaggle model weights are placed in `/kaggle/temp`, which is scratch space, not in the 20 GB saved `/kaggle/working` output allowance. Only selected audio WAVs and manifests should be preserved or copied to Drive. New sessions will need the model weights again unless you deliberately create a separate private cache dataset after checking Kaggle storage limits.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "qwen-tts", "soundfile"], check=True)
audio_dir = ROOT / "video/ninefold_tide/audio"
work = ROOT / "video/ninefold_tide/workspace/04_voice_foley_music"
subprocess.run([sys.executable, str(audio_dir/"voice_pipeline.py"), "prepare", "--workspace", str(work)], check=True)
subprocess.run([sys.executable, str(audio_dir/"voice_pipeline.py"), "audit", "--stage", "structure", "--workspace", str(work)], check=True)
subprocess.run([sys.executable, str(audio_dir/"qwen_audio_worker.py"), "gpu-check", "--workspace", str(work)], check=True)


## 2. Create ORIGINAL speaking-voice references for Naya, Kito and Iri

This is the first **large** model download (approximately 4.5 GB). It runs once, synthesizing each speaker from the canonical character descriptions, not anyone else's voice. The recording has **not** passed auditions yet.

In [ ]:
# Running this cell explicitly authorizes the high-data download ON THE KAGGLE RUNTIME.
subprocess.run([sys.executable, str(audio_dir/"qwen_audio_worker.py"), "design",
                "--characters", "naya,kito,iri", "--workspace", str(work), "--allow-download"], check=True)
for person in ("naya","kito","iri"):
    path = work / "references" / f"{person}_unapproved.wav"
    print("UNAPPROVED actor candidate:", person, path.name)
    display(Audio(filename=str(path)))


## 3. Review the references BEFORE making all dialogue

Listen for genuine teenage vs preteen resonance, a distinct Naya/Kito/Iri identity, credible accent and no famous-actor resemblance. If any anchor sounds wrong, do not proceed; regenerate that actor and audit its SHA and transcript. The file status remains unapproved. Advance only after selecting suitable reference candidates.

In [ ]:
from pathlib import Path
for person in ("naya","kito","iri"):
    path = work/"references"/f"{person}_unapproved.wav"
    assert path.exists() and path.stat().st_size > 1024
print("Audio reference files exist. LISTENING and production approval are still required.")


## 4. Generate three pilot takes per line plus name-pronunciation drills

This loads the separate Base model (another large model download, approximately 4.5 GB). It preserves the anchor files and writes unapproved WAV auditions and SHA-256 review manifests. Never say an emotion was performed correctly simply because the seed changed.

In [ ]:
# Running this cell explicitly authorizes Base model download on Kaggle
subprocess.run([sys.executable, str(audio_dir/"qwen_audio_worker.py"), "perform",
                "--characters", "naya,kito,iri", "--takes-per-line", "3",
                "--name-drills", "--workspace", str(work), "--allow-download"], check=True)
import json
manifest = json.loads((work/"reviews"/"auditions_qwen-base.json").read_text())
for item in manifest["candidates"]:
    print(item["line_id"], item["cast"], "take", item["take"], "seconds", item.get("duration_seconds"),
          "fits director's old shot:", item.get("fits_existing_shot"), item.get("file", ""))


## 5. Listen in dramatic context and store the best candidates

Listen to several adjacent lines in sequence, focusing on timing, authentic breath, emotional variation and accurate names. A natural take that exceeds a shot's current duration should lead to an **approved editorial retime**, not audio speed-up. Compare Chatterbox separately for lines with poor emotion. These files are not a final soundtrack.

In [ ]:
from IPython.display import display, Audio
for item in manifest["candidates"][:9]:
    path = item.get("file")
    if path and pathlib.Path(path).exists():
        print(item["line_id"], item["cast"], item["take"])
        display(Audio(filename=path))


## 6. Export ONLY audition WAVs and metadata for director review

This ZIP is review material, not a commercial production master. Download/upload small selected audio files to the project's existing Google Drive folder; never upload gigabytes of model caches unnecessarily.

In [ ]:
import zipfile
zip_path = pathlib.Path("/kaggle/working/ninefold_voice_auditions_UNAPPROVED.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("references","takes","reviews"):
        for f in (work/sub).rglob("*"):
            if f.is_file():
                z.write(f, arcname=f.relative_to(work))
    for filename in ("cast.json","pronunciation_lexicon.json","pilot_voice_lines.json"):
        z.write(work/filename, arcname=filename)
print(zip_path, "bytes", zip_path.stat().st_size, "UNAPPROVED")
